## Observações

- O dataset é composto por uma coluna de ID, uma coluna alvo e inúmeras colunas de loci
- As colunas loci são nomeadas como `chr{cromossomo}_{locus}` onde cromossomo é o número do cromossomo e locus é a posição no cromossomo
- Toda coluna numérica pode ser convertida para `int8`, o que reduz o tamanho do dataset de 11,2 MB para 3,6 MB

## Decisões

- Converter todas as colunas numéricas para `int8`.
- Dropar a coluna `id`.

### Priorizar seleção de features

- Aplicar um filtro barato inicial, como `SelectKBest`, para reduzir o número de loci para algumas centenas (500?).
- Aplicar `RFE` com estimador linear regularizado.
- Avaliar a estabilidade da seleção com `StratifiedShuffleSplit`.
- Treinar posteriormente um novo modelo, como `XGBoost`, usando somente os loci estáveis.
- Aplicar SHAP ao modelo final.

Sugestão de um script para seleção de features:

```py
mutual_information = partial(
    mutual_info_classif, discrete_features=True, random_state=42
)
selector_estimator = LogisticRegression(
    penalty="l2", class_weight="balanced", max_iter=5_000, random_state=42
)

selection_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("filter", SelectKBest(score_func=mutual_information, k=500)),
    ("rfe", RFE(
        estimator=selector_estimator, n_features_to_select=50, step=0.01
    )),
])

selection_count = Counter()
splitter = StratifiedShuffleSplit(n_splits=30, test_size=0.20, random_state=42)

for train_idxs, _ in splitter.split(X, y):
    # ...
    selection_count.update(selected_features)

stability = (
    pd.Series(selection_count, name="selection_count")
    .rename_axis("feature")
    .to_frame()
)
stability["selection_rate"] = stability["selection_count"] / splitter.n_splits
stability = stability.sort_values("selection_rate", ascending=False)
```

In [ ]:
import pandas as pd

from covid.common.paths import RAW_DATA_PATH

raw_data = pd.read_csv(RAW_DATA_PATH)
raw_data.shape

In [ ]:
raw_data.sample(10)

In [ ]:
raw_data.info()

In [ ]:
raw_data.columns

In [ ]:
[col for col in raw_data.columns if not str(col).startswith("chr")]

In [ ]:
raw_data.select_dtypes(include="str").columns.to_list()

In [ ]:
# Number of locus in each chromosome
chr_counts = (
    raw_data.columns
    .to_series()
    .str.extract(r"^(chr[^_]+)", expand=False)
    .value_counts()
    .sort_index()
)
chr_counts

In [ ]:
def can_convert_to_int(series: pd.Series) -> bool:
    numeric = pd.to_numeric(series, errors="coerce")

    has_non_numeric = (numeric.isna() & series.notna()).any()
    if has_non_numeric:
        return False

    all_values_are_int = numeric.dropna().mod(1).eq(0).all()
    return all_values_are_int


non_integer_columns = [
    column
    for column in raw_data.columns
    if not can_convert_to_int(raw_data[column])
]

non_integer_columns

In [ ]:
raw_data.select_dtypes(include="number").astype("int8", errors="ignore").info()